In [1]:
# WEATHER FEATURE ENGINEERING - USING ISLAMABAD AS REFERENCE CITY
# Adding Temperature, Humidity, and Rainfall Flag features using historical weather data as a reference-city proxy

import pandas as pd
import requests

In [2]:
# 1. LOAD CLEANED DATASET

df_original = pd.read_csv("../Data/Cleaned/Cleaned Readable Data.csv")
df_original["Date"] = pd.to_datetime(df_original["Date"])

# Working on a copy so the original cleaned dataset stays untouched
df = df_original.copy()

print(f"Dataset loaded: {len(df):,} rows, {len(df.columns)} columns")
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
df.head()

Dataset loaded: 51,862 rows, 9 columns
Date range: 2020-01-01 to 2025-11-30


,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production
0,2025-11-30,21,22,Wind,334,Sunday,November,Fall,5281
1,2025-11-30,18,19,Wind,334,Sunday,November,Fall,3824
2,2025-11-30,16,17,Wind,334,Sunday,November,Fall,3824
3,2025-11-30,23,0,Wind,334,Sunday,November,Fall,6120
4,2025-11-30,6,7,Wind,334,Sunday,November,Fall,4387


In [3]:
# 2. FETCH HISTORICAL WEATHER DATA FROM OPEN-METEO WEATHER API FOR ISLAMABAD

# Islamabad coordinates
LATITUDE = 33.6844
LONGITUDE = 73.0479

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": df["Date"].min().strftime("%Y-%m-%d"),
    "end_date": df["Date"].max().strftime("%Y-%m-%d"),
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,windspeed_10m",
    "timezone": "Asia/Karachi"
}

response = requests.get(url, params=params)
response.raise_for_status()
weather_json = response.json()

print("Weather data fetched successfully")
print("Hourly records returned:", len(weather_json["hourly"]["time"]))

Weather data fetched successfully
Hourly records returned: 51864


In [4]:
# 3. BUILD WEATHER DATAFRAME

weather_df = pd.DataFrame(weather_json["hourly"])
weather_df["time"] = pd.to_datetime(weather_df["time"])

weather_df.rename(columns={
    "time": "Datetime",
    "temperature_2m": "Temperature_C",
    "relative_humidity_2m": "Humidity_Percent",
    "precipitation": "Precipitation_mm",
    "windspeed_10m": "WindSpeed_kmh"
}, inplace=True)

# Rainfall flag: Yes if any precipitation was recorded in that hour, No otherwise
weather_df["Rainfall_Flag"] = weather_df["Precipitation_mm"].apply(lambda x: "Yes" if x > 0 else "No")

print(f"Weather dataframe built: {len(weather_df):,} hourly rows")
weather_df.head()

Weather dataframe built: 51,864 hourly rows


,Datetime,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh,Rainfall_Flag
0,2020-01-01 00:00:00,4.1,84,0.0,5.1,No
1,2020-01-01 01:00:00,4.0,85,0.0,5.5,No
2,2020-01-01 02:00:00,4.0,83,0.0,4.1,No
3,2020-01-01 03:00:00,3.0,88,0.0,4.4,No
4,2020-01-01 04:00:00,3.6,86,0.0,4.6,No


In [5]:
# 4. CHECK FOR MISSING VALUES IN WEATHER DATA

print("Missing values per weather column:")
print(weather_df.isnull().sum())

Missing values per weather column:
Datetime            0
Temperature_C       0
Humidity_Percent    0
Precipitation_mm    0
WindSpeed_kmh       0
Rainfall_Flag       0
dtype: int64


In [6]:
# 5. MERGE WEATHER DATA WITH MAIN DATASET

# Build a matching hourly datetime key using Date + Start_Hour
df["Datetime"] = df["Date"] + pd.to_timedelta(df["Start_Hour"], unit="h")

df_with_weather = df.merge(weather_df, on="Datetime", how="left")

print(f"Merged dataset shape: {df_with_weather.shape}")
print("\nMissing values introduced by merge:")
print(df_with_weather[["Temperature_C", "Humidity_Percent", "Precipitation_mm", "WindSpeed_kmh", "Rainfall_Flag"]].isnull().sum())
df_with_weather.head()

Merged dataset shape: (51862, 15)

Missing values introduced by merge:
Temperature_C       0
Humidity_Percent    0
Precipitation_mm    0
WindSpeed_kmh       0
Rainfall_Flag       0
dtype: int64


,Date,Start_Hour,End_Hour,Source,Day_of_Year,Day_Name,Month_Name,Season,Production,Datetime,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh,Rainfall_Flag
0,2025-11-30,21,22,Wind,334,Sunday,November,Fall,5281,2025-11-30 21:00:00,11.8,72,0.0,5.9,No
1,2025-11-30,18,19,Wind,334,Sunday,November,Fall,3824,2025-11-30 18:00:00,13.5,67,0.0,7.2,No
2,2025-11-30,16,17,Wind,334,Sunday,November,Fall,3824,2025-11-30 16:00:00,17.3,50,0.0,5.1,No
3,2025-11-30,23,0,Wind,334,Sunday,November,Fall,6120,2025-11-30 23:00:00,10.4,69,0.0,5.4,No
4,2025-11-30,6,7,Wind,334,Sunday,November,Fall,4387,2025-11-30 06:00:00,8.2,62,0.0,1.6,No


In [7]:
# 6. HANDLE ANY MERGE GAPS

# Drop the helper Datetime column as it is no longer needed after merging
df_with_weather.drop(columns=["Datetime"], inplace=True)

# If any weather values are missing (e.g. edge-of-range timestamps), fill using forward/backward fill since weather changes gradually hour to hour
weather_cols = ["Temperature_C", "Humidity_Percent", "Precipitation_mm", "WindSpeed_kmh"]
df_with_weather[weather_cols] = df_with_weather[weather_cols].ffill().bfill()
df_with_weather[weather_cols] = df_with_weather[weather_cols].ffill().bfill()
df_with_weather["Rainfall_Flag"] = df_with_weather["Rainfall_Flag"].fillna("No")

print("Remaining missing values after fill:")
print(df_with_weather[weather_cols + ["Rainfall_Flag"]].isnull().sum())

Remaining missing values after fill:
Temperature_C       0
Humidity_Percent    0
Precipitation_mm    0
WindSpeed_kmh       0
Rainfall_Flag       0
dtype: int64


In [8]:
# 7. VERIFICATION OF THE NEW FEATURES

print("Temperature range (C):", df_with_weather["Temperature_C"].min(), "to", df_with_weather["Temperature_C"].max())
print("Humidity range (%):", df_with_weather["Humidity_Percent"].min(), "to", df_with_weather["Humidity_Percent"].max())
print("Wind Speed range (km/h):", df_with_weather["WindSpeed_kmh"].min(), "to", df_with_weather["WindSpeed_kmh"].max())
print("\nRainfall flag distribution:")
print(df_with_weather["Rainfall_Flag"].value_counts())

df_with_weather.describe()

Temperature range (C): 1.8 to 44.3
Humidity range (%): 7 to 100
Wind Speed range (km/h): 0.0 to 40.8

Rainfall flag distribution:
Rainfall_Flag
No     46073
Yes     5789
Name: count, dtype: int64


,Date,Start_Hour,End_Hour,Day_of_Year,Production,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh
count,51862,51862.000000,51862.000000,51862.000000,51862.000000,51862.000000,51862.000000,51862.000000,51862.000000
mean,2022-12-16 01:07:56.603293184,11.499711,11.499672,180.800278,6215.242625,21.592575,59.241256,0.124289,7.248243
min,2020-01-01 00:00:00,0.000000,0.000000,1.000000,58.000000,1.800000,7.000000,0.000000,0.000000
25%,2021-06-24 00:00:00,5.250000,5.250000,91.000000,3111.000000,15.000000,45.000000,0.000000,4.700000
50%,2022-12-16 00:00:00,11.000000,11.000000,181.000000,5372.000000,22.600000,60.000000,0.000000,6.600000
75%,2024-06-08 00:00:00,17.000000,17.000000,271.000000,8501.000000,27.700000,75.000000,0.000000,9.100000
max,2025-11-30 00:00:00,23.000000,23.000000,366.000000,23446.000000,44.300000,100.000000,48.600000,40.800000
std,NaN,6.922230,6.922186,104.292759,3978.339604,8.110513,19.983908,0.721164,3.856219


In [9]:
# 8. SAVE MODIFIED DATASET

df_with_weather.to_csv("../Data/Modified Dataset/Dataset with Weather features.csv", index=False)

print("Saved: Data/Modified Dataset/Dataset with Weather features.csv")
print(f"Final shape: {df_with_weather.shape}")
print("\nFinal columns:")
print(df_with_weather.columns.tolist())

Saved: Data/Modified Dataset/Dataset with Weather features.csv
Final shape: (51862, 14)

Final columns:
['Date', 'Start_Hour', 'End_Hour', 'Source', 'Day_of_Year', 'Day_Name', 'Month_Name', 'Season', 'Production', 'Temperature_C', 'Humidity_Percent', 'Precipitation_mm', 'WindSpeed_kmh', 'Rainfall_Flag']


**Note:** Weather values are sourced from Islamabad as a reference city, since the original dataset does not specify the actual location of the wind/solar sources. This is just added as a proxy to test whether real weather features improve model performance over time-only features.